In [1]:
import pandas as pd

# 1. Load the original JHU updates dataset
# (Replace with your local file path if already downloaded)
url = "https://raw.githubusercontent.com/CSSEGISandData/measles_data/main/measles_county_all_updates.csv"
df = pd.read_csv(url)

# Standardize column headers to lowercase to avoid syntax errors
df.columns = df.columns.str.lower()

# 2. Clean and format FIPS codes (location_id)
# Ensures 5-digit zero-padding (e.g., 4013 -> '04013')
df['location_id'] = (
    df['location_id']
    .fillna('')
    .astype(str)
    .str.split('.')
    .str[0]
    .str.zfill(5)
)

# 3. Ensure numeric case values
df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0)

# 4. Group by County identifiers and sum total cases
df_condensed = (
    df.groupby(['location_id', 'location_name', 'location_type'], as_index=False)['value']
    .sum()
    .rename(columns={'value': 'total_confirmed_cases'})
)

# 5. Sort by counties with the highest outbreak counts
df_condensed = df_condensed.sort_values(by='total_confirmed_cases', ascending=False).reset_index(drop=True)

# Inspect the result
print(f"Original file rows: {len(df)}")
print(f"Condensed file rows: {len(df_condensed)} (1 row per county)")
print(df_condensed.head(10))

# 6. Save as a new CSV file
df_condensed.to_csv("jhu_measles_county_totals.csv", index=False)
print("\nNew CSV created successfully: 'jhu_measles_county_totals.csv'")


Original file rows: 1283
Condensed file rows: 342 (1 row per county)
  location_id                    location_name location_type  \
0       45083      Spartanburg, South Carolina        county   
1       48165                    Gaines, Texas        county   
2       04015                  Mohave, Arizona        county   
3       00000  Southwest Health District, Utah        county   
4       00000         Central Region, Virginia        county   
5       48229                  Hudspeth, Texas        county   
6       49049                       Utah, Utah        county   
7       00000                    Central, Utah        county   
8       42071          Lancaster, Pennsylvania        county   
9       12021                 Collier, Florida        county   

   total_confirmed_cases  
0                    940  
1                    414  
2                    285  
3                    267  
4                    156  
5                    136  
6                    116  
7         